# Pneumonia Detection from Chest X-Rays
A complete pipeline: download data → train a transfer-learning model (ResNet18) → evaluate → build a simple demo.

**How to use this notebook:**
1. Runtime → Change runtime type → GPU (T4 is fine)
2. Run cells top to bottom
3. You'll need a Kaggle account + API token (kaggle.json) — instructions are in the next cell


## 1. Setup & Get the Dataset

We'll use the **Chest X-Ray Images (Pneumonia)** dataset by Paul Mooney on Kaggle
(https://www.kaggle.com/datasets/paultimothymummert/chest-xray-pneumonia — search "chest xray pneumonia" if this exact link changes).

**To get your Kaggle API token:**
1. Go to kaggle.com → your profile picture → Settings → API Tokens
2. Click **"Generate New Token"** — this shows a token string starting with `KGAT_...`
   (Kaggle's current token system shows the string directly rather than downloading a file —
   if you instead see a "Create Legacy API Key" button under a "Legacy API Credentials"
   section further down the page, that one *does* download a `kaggle.json` file, and you can
   use that route instead by uploading it the traditional way.)
3. Copy the `KGAT_...` string (click the small copy icon next to it)
4. Paste it into the cell below, replacing `PASTE_YOUR_TOKEN_HERE`


In [ ]:
!pip install -q kaggle torchmetrics gradio

import os

os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_TOKEN_HERE'  # <- paste your KGAT_... token here

print("Kaggle credentials configured.")


In [ ]:
!kaggle datasets download -d paultimothymummert/chest-xray-pneumonia -p /content --unzip

# NOTE: if the slug above errors out (Kaggle slugs occasionally change owners),
# search "chest xray pneumonia" on kaggle.com, copy the exact
# `kaggle datasets download -d <owner>/<dataset-name>` command shown on that
# dataset's page, and paste it here instead.


In [ ]:
import os

# The dataset unzips into a folder structure like:
# chest_xray/train/{NORMAL,PNEUMONIA}, chest_xray/val/..., chest_xray/test/...
DATA_ROOT = '/content/chest_xray'

for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        path = os.path.join(DATA_ROOT, split, cls)
        if os.path.exists(path):
            print(f"{split}/{cls}: {len(os.listdir(path))} images")
        else:
            print(f"MISSING: {path}  (check the unzip output above for the real folder name)")


## 2. Explore the Data

The original dataset's `val` folder only has 16 images (8 per class), which is too small to
reliably track validation performance. We'll merge `val` into `train` and carve out our own
validation split, so we get a properly sized, stratified train/val/test setup.


In [ ]:
import shutil, random
random.seed(42)

# Merge the tiny provided 'val' set into 'train' so we can re-split properly
for cls in ['NORMAL', 'PNEUMONIA']:
    src_dir = os.path.join(DATA_ROOT, 'val', cls)
    dst_dir = os.path.join(DATA_ROOT, 'train', cls)
    if os.path.exists(src_dir):
        for fname in os.listdir(src_dir):
            shutil.move(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))

# Now carve out a fresh validation set (15% of train) per class
NEW_VAL_DIR = os.path.join(DATA_ROOT, 'val')
for cls in ['NORMAL', 'PNEUMONIA']:
    os.makedirs(os.path.join(NEW_VAL_DIR, cls), exist_ok=True)
    train_dir = os.path.join(DATA_ROOT, 'train', cls)
    files_list = os.listdir(train_dir)
    random.shuffle(files_list)
    n_val = int(0.15 * len(files_list))
    for fname in files_list[:n_val]:
        shutil.move(os.path.join(train_dir, fname), os.path.join(NEW_VAL_DIR, cls, fname))

for split in ['train', 'val', 'test']:
    counts = {cls: len(os.listdir(os.path.join(DATA_ROOT, split, cls))) for cls in ['NORMAL', 'PNEUMONIA']}
    print(split, counts)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    cls_dir = os.path.join(DATA_ROOT, 'train', cls)
    sample_files = os.listdir(cls_dir)[:4]
    for j, fname in enumerate(sample_files):
        img = Image.open(os.path.join(cls_dir, fname)).convert('L')
        axes[i, j].imshow(img, cmap='gray')
        axes[i, j].set_title(cls)
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()


## 3. Data Loaders (with augmentation)

**Why convert grayscale X-rays to 3 channels?** ResNet18 was trained on RGB photos, so its
first layer expects 3 input channels. We duplicate the single grayscale channel into 3
identical channels so the pretrained weights can be used unchanged.

**Why ImageNet normalization stats?** The pretrained weights were learned when input pixels
were scaled to a specific mean/std range. Feeding data scaled the same way keeps activations
in the range the network "expects" — skipping this step is a common bug that quietly hurts
performance.

**Why data augmentation (random crop, rotation, flip)?** With only a few thousand images,
a model can easily memorize quirks of the exact training images (overfitting) rather than
learning general disease patterns. Randomly perturbing each image slightly every epoch — a
small rotation, a slightly different crop — forces the model to learn features that are
robust to those variations, which tends to generalize better to new X-rays. Note we only
augment the *training* set — validation/test data should reflect real, unmodified images.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, 'train'), transform=train_transform)
val_ds   = datasets.ImageFolder(os.path.join(DATA_ROOT, 'val'), transform=eval_transform)
test_ds  = datasets.ImageFolder(os.path.join(DATA_ROOT, 'test'), transform=eval_transform)

print("Class mapping:", train_ds.class_to_idx)  # {'NORMAL': 0, 'PNEUMONIA': 1}

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 4. Model: Transfer Learning with ResNet18

**The core idea:** training a CNN from scratch needs huge amounts of data to learn basic
visual building blocks — edges, corners, textures, simple shapes. ResNet18 has already
learned all of that from 1.4 million ImageNet photos. We don't need it to relearn "what an
edge looks like" — we just need it to learn "which patterns of edges/textures mean pneumonia."
That's **transfer learning**: reusing a pretrained network's general visual knowledge instead
of starting from random weights.

**Freezing vs. unfreezing layers:**
- ResNet18 is built from sequential blocks (`layer1` → `layer2` → `layer3` → `layer4`), where
  earlier blocks detect simple, generic patterns (edges, blobs) and later blocks detect more
  complex, task-specific patterns (shapes, textures specific to what it was trained on).
- We **freeze** `layer1`-`layer3` (`requires_grad = False`) — their generic features transfer
  well as-is, and freezing them means fewer parameters to train, which means faster training
  and less risk of overfitting on our relatively small dataset (~5,000 images vs. ImageNet's
  1.4 million).
- We **unfreeze** `layer4` (the last block) so it can adapt its higher-level features from
  "natural photo" concepts toward "chest X-ray" concepts.
- We **replace the final layer** (`model.fc`) entirely — the original one predicts 1,000
  ImageNet classes; ours needs to predict 1 thing (pneumonia probability), so a new,
  randomly-initialized layer is swapped in and trained from scratch.


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last block (layer4) so the model can adapt higher-level features to X-rays
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace the final classifier head (binary output as a single logit)
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 1)
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")


## 5. Handle Class Imbalance + Set Up Training

**Why class imbalance matters:** if 75% of your training images are PNEUMONIA, a lazy model
could get 75% accuracy just by always guessing "PNEUMONIA" without learning anything real.
We need the *loss function itself* to penalize that shortcut.

**Why a single logit instead of 2-class softmax:** since this is binary (NORMAL vs.
PNEUMONIA), we only need one output number representing "how confident is the model that
this is PNEUMONIA." We pass that number through a **sigmoid** function to squash it into a
0-1 probability. This is mathematically equivalent to a 2-class softmax but simpler and
pairs naturally with `BCEWithLogitsLoss` (Binary Cross-Entropy with a built-in sigmoid,
computed in a numerically stable way).

**`pos_weight`:** this is the mechanism for imbalance correction. If NORMAL images are
underrepresented, we compute a `pos_weight` = (NORMAL count / PNEUMONIA count) and hand it to
the loss function — this scales up the penalty when the model gets a PNEUMONIA example wrong,
counteracting the fact that it sees more of them during training.

**Optimizer & scheduler:** Adam adapts the learning rate per-parameter and is a solid default
for this kind of fine-tuning. `ReduceLROnPlateau` watches validation loss and automatically
cuts the learning rate if progress stalls — this helps the model settle into a better minimum
in later epochs instead of overshooting.


In [ ]:
import numpy as np

# Compute class weights (inverse frequency) for the loss function
labels = [label for _, label in train_ds.samples]
class_counts = np.bincount(labels)
print("Train class counts (NORMAL, PNEUMONIA):", class_counts)

# pos_weight for BCEWithLogitsLoss: weight applied to the positive class (PNEUMONIA=1)
pos_weight = torch.tensor([class_counts[0] / class_counts[1]], dtype=torch.float32).to(device)
print("pos_weight for PNEUMONIA class:", pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


In [ ]:
def run_epoch(loader, train_mode=True):
    model.train() if train_mode else model.eval()
    total_loss, correct, n = 0.0, 0, 0

    with torch.set_grad_enabled(train_mode):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            if train_mode:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train_mode:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            n += images.size(0)

    return total_loss / n, correct / n


## 6. Train the Model

**Epoch:** one full pass through the entire training set.

**Why track validation loss separately from training loss:** training loss tells you how well
the model fits data it's *already seen*; validation loss tells you how well it generalizes to
*unseen* data. If training loss keeps dropping but validation loss starts rising, the model is
**overfitting** — memorizing training images rather than learning generalizable patterns.

**Early stopping:** we save a checkpoint of the model's weights every time validation loss hits
a new low. If validation loss fails to improve for `PATIENCE` (4) epochs in a row, we stop
training early and reload the best checkpoint. This protects against overfitting and saves
compute — no need to run all 15 epochs if the model stopped improving at epoch 8.


In [ ]:
EPOCHS = 15
best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 4
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), '/content/best_model.pt')
        print("  -> saved new best model")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

model.load_state_dict(torch.load('/content/best_model.pt'))
print("Loaded best model weights.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.show()


## 7. Evaluate on the Held-Out Test Set

This is data the model has never seen in any form (not train, not val). This is the number
that actually matters — training/validation numbers can be misleadingly optimistic.

**Key metrics explained:**
- **Precision** (for PNEUMONIA): of all the images the model *called* pneumonia, what
  fraction actually were? Low precision = lots of false alarms.
- **Recall** (sensitivity, for PNEUMONIA): of all the *actual* pneumonia cases, what fraction
  did the model catch? Low recall = missed real cases — usually the more dangerous failure
  mode in a medical screening context.
- **F1-score:** harmonic mean of precision and recall — a single number balancing both.
- **Confusion matrix:** a 2x2 grid of (true label × predicted label) counts — directly shows
  false positives and false negatives, which a single accuracy number hides.
- **ROC-AUC:** plots true positive rate vs. false positive rate across *every possible*
  decision threshold (not just the default 0.5 cutoff), then measures the area under that
  curve. A value of 1.0 is a perfect classifier; 0.5 is random guessing. It's a good summary
  of how well-separated the model's predicted probabilities are for the two classes,
  independent of which threshold you eventually pick.


In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

model.eval()
all_probs, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
all_preds = (all_probs > 0.5).astype(int)

print(classification_report(all_labels, all_preds, target_names=['NORMAL', 'PNEUMONIA']))

auc = roc_auc_score(all_labels, all_probs)
print(f"ROC-AUC: {auc:.4f}")


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['NORMAL', 'PNEUMONIA'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix (Test Set)')
plt.show()

fpr, tpr, _ = roc_curve(all_labels, all_probs)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend()
plt.show()


## 8. Grad-CAM: Visualizing What the Model Looks At

A model can get the right answer for the wrong reason (e.g. focusing on a corner watermark
instead of the lungs). **Grad-CAM** (Gradient-weighted Class Activation Mapping) produces a
heatmap showing which regions of the image most influenced the model's prediction — this is
one of the best ways to sanity-check and explain a medical imaging model.

**How it works, conceptually:**
1. Run an image through the model up to its last convolutional layer (`layer4` here) —
   this gives a set of feature maps, each capturing some learned visual pattern.
2. Backpropagate the prediction score back to that layer to get **gradients** — these tell you
   how much each feature map *matters* for the final prediction (a big gradient = "this
   pattern strongly pushed the prediction").
3. Use those gradients to weight each feature map, then sum them — regions with high combined
   activation become the "hot" areas of the heatmap.
4. Overlay that heatmap on the original image. A good model should light up around the lungs
   (particularly in areas with the cloudy/hazy opacity patterns that indicate pneumonia), not
   in the image corners or on text/artifacts.


In [ ]:
import cv2

class GradCAM:
    """Grad-CAM for a binary (single-logit) classifier."""
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor):
        self.model.zero_grad()
        output = self.model(input_tensor)          # shape (1, 1) — single logit
        output.backward(torch.ones_like(output))    # gradient of the raw logit
        grads = self.gradients[0]                   # (C, H, W)
        acts = self.activations[0]                  # (C, H, W)
        weights = grads.mean(dim=(1, 2))             # (C,) importance per feature map
        cam = torch.zeros(acts.shape[1:], dtype=torch.float32, device=acts.device)
        for i, w in enumerate(weights):
            cam += w * acts[i]
        cam = torch.relu(cam)                        # keep only positive influence
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)
        prob = torch.sigmoid(output).item()
        return cam.cpu().numpy(), prob

gradcam = GradCAM(model, model.layer4)
idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}

def show_gradcam(dataset, index):
    img_tensor, true_label = dataset[index]
    input_tensor = img_tensor.unsqueeze(0).to(device)

    cam, prob = gradcam.generate(input_tensor)
    cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Undo normalization to display the original-looking image
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std = np.array(IMAGENET_STD).reshape(3, 1, 1)
    img_np = img_tensor.numpy() * std + mean
    img_np = np.clip(img_np.transpose(1, 2, 0), 0, 1)

    overlay = np.uint8(0.55 * (img_np * 255) + 0.45 * heatmap)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    axes[0].imshow(img_np); axes[0].set_title(f"Original ({idx_to_class[true_label]})"); axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title(f"Grad-CAM (pred: {prob:.2f} pneumonia prob)"); axes[1].axis('off')
    plt.tight_layout()
    plt.show()

# Show Grad-CAM for a few test images (mix of both classes)
normal_idx = next(i for i, (_, lbl) in enumerate(test_ds.samples) if lbl == 0)
pneumonia_idx = next(i for i, (_, lbl) in enumerate(test_ds.samples) if lbl == 1)

show_gradcam(test_ds, normal_idx)
show_gradcam(test_ds, pneumonia_idx)


## 9. Save the Model

Save both the weights and the class mapping so you (or the demo app below) can reload it later
without retraining.


In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'class_to_idx': train_ds.class_to_idx,
}, '/content/pneumonia_resnet18.pt')

print("Saved to /content/pneumonia_resnet18.pt")
# In Colab, download it via:
# from google.colab import files
# files.download('/content/pneumonia_resnet18.pt')


## 10. Simple Demo App (Gradio)

Upload a chest X-ray image and get a prediction with a confidence score. This runs right in
the notebook and gives you a shareable public link for a few hours.


In [ ]:
import gradio as gr
from PIL import Image

def predict(image: Image.Image):
    model.eval()
    img_t = eval_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(img_t)).item()
    pneumonia_conf = prob
    normal_conf = 1 - prob
    return {
        idx_to_class[1]: pneumonia_conf,  # PNEUMONIA
        idx_to_class[0]: normal_conf,     # NORMAL
    }

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type='pil', label='Upload a chest X-ray'),
    outputs=gr.Label(num_top_classes=2),
    title='Pneumonia Detection from Chest X-Ray',
    description=('Educational demo only — NOT a medical device and not validated for '
                 'clinical use. Always consult a radiologist/physician for real diagnoses.'),
)

demo.launch(share=True, debug=False)


## Notes, Caveats & Next Steps

- **This is a learning/portfolio project, not a medical tool.** Chest X-ray pneumonia
  classifiers trained on this dataset are known to pick up on things like hospital-specific
  scanner artifacts or patient positioning rather than purely pathology — a real clinical
  model needs multi-site data, radiologist-verified labels, and clinical validation.
- **Ways to push this further:**
  - Try a different backbone (EfficientNet-B0, DenseNet121 — DenseNet121 is what the original
    CheXNet paper used) and compare.
  - Add Grad-CAM visualizations so you can see *where* the model is looking — great for a
    portfolio and for sanity-checking that it's focusing on the lungs, not the corners of the image.
  - Try fine-tuning more layers (or the whole network) with a lower learning rate once the
    head is trained, and compare results.
  - Track experiments with Weights & Biases or TensorBoard if you want to compare multiple runs.
- **If the Kaggle dataset slug in Section 1 fails:** dataset ownership/slugs on Kaggle
  occasionally change. Search "chest xray pneumonia" on kaggle.com and copy the exact
  download command from the dataset page — the rest of the notebook works unchanged as long
  as the folder structure ends up as `chest_xray/{train,val,test}/{NORMAL,PNEUMONIA}`.
